> 🔑 **SOLUTION EDITION.** This copy contains worked answers to every exercise. Try the student notebook first - you learn by struggling a little!

In [1]:
# (Solution edition only) make the course helper module importable from this folder
import sys; sys.path.insert(0, '../notebooks')

# 🏆 Capstone: The Agricultural Intelligence Brief
### *Combine everything. Make a recommendation a Cabinet Secretary can act on.*

**⏱ about 5–6 hours (spread over several days) · Uses:** *all* of Phase 5: K-Means, hierarchical clustering, DBSCAN, PCA, association rules, evaluation, profiling, communication

---

## 📨 The brief

> **From:** Dr. Wanjiku Kariuki, Head, Agricultural Transformation Unit
> **To:** You, Lead Data Scientist
> **Subject:** One integrated brief for next month's budget committee
>
> *The Cabinet Secretary is allocating next year's agricultural support budget across **four levers**: (1) farmer programmes, (2) market infrastructure, (3) commodity-specific policy, and (4) input-supply (agro-dealer) schemes. Until now, four separate teams studied these separately. **I need one brief that connects them**: who are our farmers, what do they grow, where and how do they sell, what inputs do they buy, and **where should the money go?**
> I will give you **one page** to persuade the Committee. Make the numbers count.*

## 🔗 Why this is different from Projects 1–4
The projects each answered **one** question. Real decisions need **connected evidence**:

```
   FARMERS  ──grow──▶  PRODUCTS  ──sold at──▶  MARKETS
      │                                           │
      └──────────buy────▶  INPUT BUNDLES ◀───────┘
                    (association rules)
```

The most valuable findings come from the **links**. Examples: *Which farmer segments depend on perishable crops?* *Which segments live far from markets?* *Which input bundles serve which segment?*

## 🗺️ Milestones

| # | Milestone | Data | Methods |
|---|---|---|---|
| 1 | **Farmer segments** | `farmers` | impute, log, scale, K-Means, profile |
| 2 | **Commodity groups** | `products` | log, scale, Ward |
| 3 | **Link farmers → commodities** (test a hypothesis!) | both | crosstab, groupby |
| 4 | **Market structure** | `markets` | K-Means, DBSCAN (haversine), PCA index |
| 5 | **Input bundles** | `agro_input_baskets` | Apriori, rules |
| 6 | **Integrated strategy table** | all | synthesis |
| 7 | **The one-page brief** | all | communication |

> 💡 **You may: and should: reuse your code from Projects 1–4.** The skill being graded here is **integration and judgement**, not re-typing.

## 📏 Grading rubric (100 points)

| Criterion | Pts | Excellent looks like |
|---|---:|---|
| **Technical soundness** | 25 | Correct preprocessing; *k* justified by multiple evidence; DBSCAN uses radians/haversine; rules filtered by lift |
| **Integration** | 20 | At least **two cross-dataset links**, with one **hypothesis tested** honestly (including if it fails) |
| **Insight quality** | 15 | Findings are *non-obvious*, numerically specific, and surprising where appropriate |
| **Strategy table** | 15 | Every segment has a constraint, a market channel, an input bundle and a programme: all tied to evidence |
| **The brief** | 20 | One page; plain language; numbers; clear *decision requested*; honest limitations |
| **Reproducibility** | 5 | Notebook runs top-to-bottom; assumptions stated |

In [2]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import haversine_distances
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from course_utils import set_style, load, check, profile_clusters, PALETTE
import warnings; warnings.filterwarnings("ignore")
set_style()
RANDOM_STATE = 42; EARTH_R = 6371.0

farmers, products, markets = load("farmers"), load("products"), load("markets")
agro = load("agro_input_baskets")
print({k: v.shape for k, v in dict(farmers=farmers, products=products, markets=markets, agro=agro).items()})

{'farmers': (700, 15), 'products': (48, 11), 'markets': (103, 15), 'agro': (7384, 2)}


## Milestone 1 · Farmer segments 🌽

Reuse your Project 2 pipeline. (Features: production practices only; median-impute; `log1p` farm size; scale; K-Means with a *k* you can defend.)

### 🧩 Milestone 1: Segment the farmers
**Difficulty:** ⭐⭐  |  **Time:** ~30 min

Produce:
- `f_labels`: K-Means cluster ids (array of length 700), with `f_k` clusters (3–5)
- `f_names`: dict `{cluster_id: unique segment name}`
- `fdf`: farmers (numeric NaNs filled) with columns `cluster` and `segment`
- `f_profile`: `profile_clusters(fdf, f_labels, feature_cols)` (original units)
- `f_sizes`: Series: % of farmers per segment (sums to 100)

*(Recall: exclude revenue, county, crop, ids.)*

<details><summary>💡 Hint 1</summary>

Copy your Project 2 solution: features `farm_size_acres, yield_bags_per_acre, fertilizer_kg_per_acre, irrigated, labour_days_per_acre, improved_seed, coop_member, distance_to_market_km, post_harvest_loss_pct`.

</details>
<details><summary>💡 Hint 2</summary>

Name segments from the relative profile: huge + low labour → commercial grain; irrigated + labour-intensive + high yield → horticulture; tiny + high loss + low fertiliser → subsistence; the rest → dairy/mixed co-op.

</details>
<details><summary>💡 Hint 3</summary>

`fdf['segment'] = fdf['cluster'].map(f_names)`

</details>

In [3]:
feature_cols = ["farm_size_acres", "yield_bags_per_acre", "fertilizer_kg_per_acre", "irrigated", "labour_days_per_acre",
                "improved_seed", "coop_member", "distance_to_market_km", "post_harvest_loss_pct"]
Xf = farmers[feature_cols].copy(); Xf = Xf.fillna(Xf.median()); Xf["farm_size_acres"] = np.log1p(Xf["farm_size_acres"])
Xf_s = StandardScaler().fit_transform(Xf)
f_k = 4                                   # (justified in Project 2: stability ~0.95, four fundable programmes)
f_labels = KMeans(f_k, n_init=10, random_state=RANDOM_STATE).fit_predict(Xf_s)

fdf = farmers.fillna(farmers.median(numeric_only=True)).copy(); fdf["cluster"] = f_labels
f_profile = profile_clusters(fdf, f_labels, feature_cols)
rel = (f_profile[feature_cols] - fdf[feature_cols].mean()) / fdf[feature_cols].std()
def name_f(r):
    if r["farm_size_acres"] > 1.0 and r["labour_days_per_acre"] < -0.5: return "Commercial grain farmers"
    if r["irrigated"] > 1.0 and r["labour_days_per_acre"] > 0.5: return "Intensive horticulture farmers"
    if r["post_harvest_loss_pct"] > 0.8 and r["fertilizer_kg_per_acre"] < -0.5: return "Subsistence smallholders"
    return "Dairy / mixed co-operative farmers"
f_names = {c: name_f(rel.loc[c]) for c in rel.index}
fdf["segment"] = fdf["cluster"].map(f_names)
f_sizes = 100 * fdf["segment"].value_counts(normalize=True)
display(f_profile.round(1)); print(f_sizes.round(1).to_dict())

,size,share_%,farm_size_acres,yield_bags_per_acre,fertilizer_kg_per_acre,irrigated,labour_days_per_acre,improved_seed,coop_member,distance_to_market_km,post_harvest_loss_pct
cluster,,,,,,,,,,,
0,252,36.0,1.9,11.8,14.0,0.0,37.6,0.3,0.1,13.8,27.4
1,136,19.4,3.4,38.0,91.4,0.9,59.2,0.9,0.6,6.4,13.7
2,112,16.0,52.3,25.6,69.1,0.1,5.8,0.9,0.6,8.0,9.2
3,200,28.6,6.3,20.5,42.7,0.1,23.5,0.7,0.6,6.3,10.8


{'Subsistence smallholders': 36.0, 'Dairy / mixed co-operative farmers': 28.6, 'Intensive horticulture farmers': 19.4, 'Commercial grain farmers': 16.0}


In [4]:
# ✔ Self-check (run me after you've written your answer)
check('3 to 5 segments, one label per farmer', lambda: 3 <= f_k <= 5 and len(f_labels) == 700 and len(set(f_labels)) == f_k)
check('unique names for every cluster', lambda: set(f_names) == set(f_labels) and len(set(f_names.values())) == f_k)
check('sizes sum to 100', lambda: abs(f_sizes.sum() - 100) < 1e-6)
check('no segment is tiny (< 5%)', lambda: f_sizes.min() > 5)

✅ 3 to 5 segments, one label per farmer
✅ unique names for every cluster
✅ sizes sum to 100
✅ no segment is tiny (< 5%)


## Milestone 2 · Commodity groups 🍅

Reuse Project 4: log-transform skewed positive columns, standardise, Ward linkage, cut at 5–8 groups, and **name the groups by reading their members**.

### 🧩 Milestone 2: Group the commodities
**Difficulty:** ⭐⭐  |  **Time:** ~25 min

Produce:
- `p_group`: Ward cluster ids (array of 48) with `p_k` groups
- `p_names`: dict `{group_id: name}`
- `crop_group`: a dict mapping **product name → group name** (for all 48 products)
- `perishable_group`: the **name** of the group containing `"Tomatoes"`

Sanity: Tomatoes and Kale should share a group; Maize should not be in it.

<details><summary>💡 Hint 1</summary>

`feature_cols_p = [c for c in products.columns if c != 'product']`; log10 the positive skewed ones: `avg_price_kes_per_kg`, `shelf_life_days`, `monthly_volume_tonnes` (and leave `export_share`, which contains zeros).

</details>
<details><summary>💡 Hint 2</summary>

`p_group = fcluster(linkage(Xp, 'ward'), p_k, 'maxclust')`

</details>
<details><summary>💡 Hint 3</summary>

Name each group after what you read in its member list, e.g. look for a telltale member: Maize → grains; Beans → pulses; Potatoes → roots; Tomatoes → perishables; Milk → livestock; Tea → cash crops.

</details>

In [5]:
feature_cols_p = [c for c in products.columns if c != "product"]
Xp = products[feature_cols_p].copy()
for c in ["avg_price_kes_per_kg", "shelf_life_days", "monthly_volume_tonnes"]:
    Xp[c] = np.log10(Xp[c])
Xp_s = StandardScaler().fit_transform(Xp)
p_k = 7
p_group = fcluster(linkage(Xp_s, "ward"), p_k, "maxclust")
telltale = {"Maize": "Storable staple grains", "Beans": "Pulses & legumes", "Potatoes": "Roots & tubers",
            "Tomatoes": "Perishable vegetables & tropical fruit", "Avocado": "Other seasonal fruit",
            "Milk": "Livestock & animal products", "Tea": "Export cash crops"}
p_names = {}
for g in sorted(set(p_group)):
    members = set(products["product"][p_group == g])
    p_names[int(g)] = next((n for k, n in telltale.items() if k in members), f"Group {g}")
crop_group = {prod: p_names[g] for prod, g in zip(products["product"], p_group)}
perishable_group = crop_group["Tomatoes"]
print(p_names); print("perishable group:", perishable_group)

{1: 'Other seasonal fruit', 2: 'Perishable vegetables & tropical fruit', 3: 'Storable staple grains', 4: 'Roots & tubers', 5: 'Pulses & legumes', 6: 'Livestock & animal products', 7: 'Export cash crops'}
perishable group: Perishable vegetables & tropical fruit


In [6]:
# ✔ Self-check (run me after you've written your answer)
check('5 to 8 groups over 48 products', lambda: 5 <= p_k <= 8 and len(p_group) == 48 and len(set(p_group)) == p_k)
check('every product is mapped to a group name', lambda: len(crop_group) == 48)
check('Tomatoes and Kale share a group', lambda: crop_group['Tomatoes'] == crop_group['Kale'])
check('Maize is in a different group', lambda: crop_group['Maize'] != crop_group['Tomatoes'])
check('group names are unique', lambda: len(set(p_names.values())) == p_k)

✅ 5 to 8 groups over 48 products
✅ every product is mapped to a group name
✅ Tomatoes and Kale share a group
✅ Maize is in a different group
✅ group names are unique


## Milestone 3 · Link farmers to commodities 🔗: **test a hypothesis**

Farmers have a `primary_crop`; products belong to groups. Join them to ask: *what do each segment's farmers grow, in terms of market behaviour?*

> 🧪 **Hypothesis from the Commodity Board:** *"Post-harvest losses are a problem of **perishable crops**. Farmers growing tomatoes, kale, etc. lose more of their harvest than other farmers. So fund cold-chain infrastructure."*
>
> **Your job is to test it with data, not to confirm it.** A good scientist reports the result even when it contradicts the boss.

### 🧩 Milestone 3: Exposure analysis and hypothesis test
**Difficulty:** ⭐⭐⭐  |  **Time:** ~30 min

1. `fdf["crop_group"] = fdf["primary_crop"].map(crop_group)`; crops not in the products table (fodder: *Napier Grass, Dairy Fodder*) → fill with `"Fodder (not traded)"`.
2. `exposure = pd.crosstab(fdf["segment"], fdf["crop_group"], normalize="index")` (each segment's crop-group mix; rows sum to 1).
3. `perishable_share` = a Series: share of each segment's farmers growing **perishable-group** crops. `most_exposed_segment` = its argmax.
4. **Test the hypothesis:** `loss_perishable` = mean `post_harvest_loss_pct` of farmers growing perishable-group crops; `loss_other` = for all others. Then `hypothesis_supported` = `loss_perishable > loss_other` (a boolean).
5. **Dig deeper:** `loss_by_segment` = mean loss per segment. Where is the *real* loss problem?

<details><summary>💡 Hint 1</summary>

`fdf['crop_group'] = fdf['primary_crop'].map(crop_group).fillna('Fodder (not traded)')`

</details>
<details><summary>💡 Hint 2</summary>

`is_per = fdf['crop_group'] == perishable_group`; `fdf.groupby('segment')[...]`: or `is_per.groupby(fdf['segment']).mean()`.

</details>
<details><summary>💡 Hint 3</summary>

If the hypothesis fails, ask *why*. Which farmers grow perishables, and what else is true of them (irrigated? co-op members? near markets?). Which farmers lose most?

</details>

In [7]:
fdf["crop_group"] = fdf["primary_crop"].map(crop_group).fillna("Fodder (not traded)")
exposure = pd.crosstab(fdf["segment"], fdf["crop_group"], normalize="index")
is_per = fdf["crop_group"] == perishable_group
perishable_share = is_per.groupby(fdf["segment"]).mean()
most_exposed_segment = perishable_share.idxmax()
loss_perishable, loss_other = fdf.loc[is_per, "post_harvest_loss_pct"].mean(), fdf.loc[~is_per, "post_harvest_loss_pct"].mean()
hypothesis_supported = bool(loss_perishable > loss_other)
loss_by_segment = fdf.groupby("segment")["post_harvest_loss_pct"].mean()
display(exposure.round(2))
print(f"perishable growers lose {loss_perishable:.1f}% vs {loss_other:.1f}% for others -> hypothesis supported? {hypothesis_supported}")
print(loss_by_segment.round(1).to_dict())
# FINDING: the hypothesis is REJECTED. Perishable-crop growers are mostly commercial, irrigated, co-op-organised horticulture farmers,
# who lose LESS (~14%) than average. The heavy losses (~28%) are among subsistence smallholders growing STORABLE grains, roots and pulses,
# i.e. a farmer-type problem (storage, handling, inputs), not a crop-perishability problem. Fund hermetic storage + extension for
# smallholders, rather than blanket cold-chain.

crop_group,Fodder (not traded),Other seasonal fruit,Perishable vegetables & tropical fruit,Pulses & legumes,Roots & tubers,Storable staple grains
segment,,,,,,
Commercial grain farmers,0.0,0.00,0.00,0.00,0.00,1.00
Dairy / mixed co-operative farmers,0.3,0.07,0.14,0.05,0.23,0.21
Intensive horticulture farmers,0.0,0.01,0.99,0.00,0.00,0.01
Subsistence smallholders,0.0,0.01,0.02,0.21,0.24,0.52


perishable growers lose 14.1% vs 18.0% for others -> hypothesis supported? False
{'Commercial grain farmers': 9.3, 'Dairy / mixed co-operative farmers': 10.8, 'Intensive horticulture farmers': 13.7, 'Subsistence smallholders': 27.4}


In [8]:
# ✔ Self-check (run me after you've written your answer)
check('exposure rows each sum to 1', lambda: np.allclose(exposure.sum(axis=1), 1))
check('the most exposed segment is horticulture', lambda: 'orticulture' in most_exposed_segment)
check('the hypothesis is NOT supported by the data', lambda: hypothesis_supported is False)
check('the highest-loss segment is the subsistence smallholders', lambda: 'ubsistence' in loss_by_segment.idxmax())

✅ exposure rows each sum to 1
✅ the most exposed segment is horticulture
✅ the hypothesis is NOT supported by the data
✅ the highest-loss segment is the subsistence smallholders


## Milestone 4 · Market structure 🏪

Reuse Project 3: **behavioural types** (K-Means on the 10 behaviour columns, k=3), **geographic regions** (DBSCAN, haversine, 40 km, min_samples 4), **isolated markets**, and a **hub per large region**.

### 🧩 Milestone 4: Market types, regions and hubs
**Difficulty:** ⭐⭐⭐  |  **Time:** ~35 min

Produce:
- `m_types`: K-Means labels (k = 3) on the scaled behaviour features
- `geo_labels`: DBSCAN labels (haversine; `eps = 40 km`; `min_samples = 4`); `n_regions`, `n_remote`
- `mk`: markets DataFrame with columns `region` and `dev_index` (PCA PC1 rescaled 0–100, oriented so higher = more developed)
- `hub_table`: one row per region with ≥ 8 markets: `region`, `n_markets`, `hub_market` (highest `dev_index`), `avg_km_to_hub`
- `remote_dist`: mean `distance_to_city_km` of the isolated markets

<details><summary>💡 Hint 1</summary>

Behaviour columns: everything numeric **except** `latitude`/`longitude`.

</details>
<details><summary>💡 Hint 2</summary>

DBSCAN: `eps=40/EARTH_R`, `metric='haversine'`, `algorithm='ball_tree'`, input `np.radians(lat/lon)`.

</details>
<details><summary>💡 Hint 3</summary>

Hub distances: `haversine_distances(radians_sub, radians_hub.reshape(1, -1)) * EARTH_R`.

</details>

In [9]:
mcols = ["avg_daily_traders", "avg_daily_volume_tonnes", "price_volatility", "distance_to_city_km", "road_quality_1_5",
         "has_cold_storage", "perishable_share", "n_commodities", "mobile_payment_share", "market_days_per_week"]
Xm = StandardScaler().fit_transform(markets[mcols])
m_types = KMeans(3, n_init=10, random_state=RANDOM_STATE).fit_predict(Xm)

coords = np.radians(markets[["latitude", "longitude"]].values)
geo_labels = DBSCAN(eps=40 / EARTH_R, min_samples=4, metric="haversine", algorithm="ball_tree").fit_predict(coords)
n_regions, n_remote = len(set(geo_labels) - {-1}), int((geo_labels == -1).sum())

mk = markets.copy(); mk["region"] = geo_labels; mk["type"] = m_types
p1 = PCA(1).fit(Xm); s = p1.transform(Xm).ravel()
if p1.components_[0, mcols.index("avg_daily_traders")] < 0: s = -s
mk["dev_index"] = 100 * (s - s.min()) / (s.max() - s.min())

rows = []
for r in sorted(set(geo_labels) - {-1}):
    sub = mk[mk["region"] == r]
    if len(sub) < 8: continue
    hub = sub.loc[sub["dev_index"].idxmax()]
    d = haversine_distances(np.radians(sub[["latitude", "longitude"]].values), np.radians(hub[["latitude", "longitude"]].values.astype(float).reshape(1, -1))) * EARTH_R
    rows.append(dict(region=r, n_markets=len(sub), hub_market=hub["market_name"], avg_km_to_hub=round(float(d.mean()), 1)))
hub_table = pd.DataFrame(rows)
remote_dist = mk.loc[mk["region"] == -1, "distance_to_city_km"].mean()
display(hub_table); print(n_regions, n_remote, round(remote_dist))

,region,n_markets,hub_market,avg_km_to_hub
0,0,22,Nairobi Urban 1,19.8
1,1,12,Mombasa Urban 2,13.6
2,2,11,Kisumu Urban 1,22.5
3,3,13,Nakuru Urban 1,21.1
4,4,19,Eldoret Urban 1,48.1
5,5,10,Meru Urban 9,14.5
6,6,9,Nyeri Urban 1,16.9


7 7 198


In [10]:
# ✔ Self-check (run me after you've written your answer)
check('3 market types over 103 markets', lambda: len(m_types) == 103 and len(set(m_types)) == 3)
check('6 to 8 regions', lambda: 6 <= n_regions <= 8)
check('5 to 12 isolated markets', lambda: 5 <= n_remote <= 12)
check('dev_index runs 0-100', lambda: abs(mk['dev_index'].min()) < 1e-9 and abs(mk['dev_index'].max() - 100) < 1e-9)
check('at least 5 hubs, all within 100 km on average', lambda: len(hub_table) >= 5 and (hub_table['avg_km_to_hub'] < 100).all())
check('isolated markets average > 100 km from a city', lambda: remote_dist > 100)

✅ 3 market types over 103 markets
✅ 6 to 8 regions
✅ 5 to 12 isolated markets
✅ dev_index runs 0-100
✅ at least 5 hubs, all within 100 km on average
✅ isolated markets average > 100 km from a city


## Milestone 5 · Input bundles 🧺

Mine the agro-dealer receipts for **bundles** (itemsets of 3+ products) with strong associations.

### 🧩 Milestone 5: Find the bundles
**Difficulty:** ⭐⭐  |  **Time:** ~20 min

1. One-hot encode the baskets of `agro`.
2. `freq = apriori(..., min_support=0.03, use_colnames=True)`; `rules = association_rules(freq, metric="confidence", min_threshold=0.5)`; keep `lift >= 2`.
3. For every rule take the **full itemset** (antecedent ∪ consequent). Deduplicate and keep only itemsets with **3 or more items**. Store them as a DataFrame `bundles` with columns `items` (a sorted tuple), `lift` (the **maximum** lift over rules for that itemset) and `support`; sorted by lift descending.
4. `n_bundles` = number of such bundles.

<details><summary>💡 Hint 1</summary>

`rules['items'] = [tuple(sorted(a | c)) for a, c in zip(rules['antecedents'], rules['consequents'])]`

</details>
<details><summary>💡 Hint 2</summary>

`bundles = rules.groupby('items').agg(lift=('lift', 'max'), support=('support', 'max')).reset_index()`; then `bundles = bundles[bundles['items'].map(len) >= 3]`.

</details>
<details><summary>💡 Hint 3</summary>

`.sort_values('lift', ascending=False).reset_index(drop=True)`

</details>

In [11]:
baskets = agro.groupby("transaction_id")["item"].apply(list).tolist()
te = TransactionEncoder(); onehot = pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_)
freq = apriori(onehot, min_support=0.03, use_colnames=True)
rules = association_rules(freq, metric="confidence", min_threshold=0.5)
rules = rules[rules["lift"] >= 2].copy()
rules["items"] = [tuple(sorted(a | c)) for a, c in zip(rules["antecedents"], rules["consequents"])]
bundles = rules.groupby("items").agg(lift=("lift", "max"), support=("support", "max")).reset_index()
bundles = bundles[bundles["items"].map(len) >= 3].sort_values("lift", ascending=False).reset_index(drop=True)
n_bundles = len(bundles)
display(bundles.round(3))

,items,lift,support
0,"(DAP Fertilizer, Fungicide, Potato Seed)",8.857,0.041
1,"(Acaricide, Dairy Meal, Mineral Lick)",7.659,0.032
2,"(Dairy Meal, Mineral Lick, Veterinary Drugs)",7.553,0.036
3,"(Knapsack Sprayer, Pesticide, Protective Gloves)",7.475,0.034
4,"(CAN Fertilizer, DAP Fertilizer, Hybrid Maize ...",4.603,0.084
5,"(DAP Fertilizer, Herbicide, Hybrid Maize Seed)",4.357,0.033
6,"(Fungicide, Knapsack Sprayer, Pesticide)",3.950,0.030
7,"(Fungicide, Pesticide, Protective Gloves)",3.899,0.039
8,"(CAN Fertilizer, Herbicide, Hybrid Maize Seed)",3.427,0.032


In [12]:
# ✔ Self-check (run me after you've written your answer)
check('at least 5 bundles of 3+ items', lambda: n_bundles >= 5)
check('every bundle has lift >= 2', lambda: (bundles['lift'] >= 2).all())
check('every bundle has support >= 0.03', lambda: (bundles['support'] >= 0.03 - 1e-9).all())
check('a maize-planting bundle exists (Hybrid Maize Seed + DAP + CAN)', lambda: any({'Hybrid Maize Seed', 'DAP Fertilizer', 'CAN Fertilizer'} <= set(i) for i in bundles['items']))

✅ at least 5 bundles of 3+ items
✅ every bundle has lift >= 2
✅ every bundle has support >= 0.03
✅ a maize-planting bundle exists (Hybrid Maize Seed + DAP + CAN)


## Milestone 6 · The integrated strategy table 🧩

This is the synthesis: **one row per farmer segment**, connecting all four datasets. Every cell should be traceable to evidence in this notebook.

| Column | Source |
|---|---|
| `segment`, `share_%` | Milestone 1 |
| `key_constraint` | Profile: loss, distance, co-op, labour… |
| `product_focus` | Milestone 3 (crop-group mix) |
| `market_channel` | Milestone 4 (market type/hub the segment should sell through) |
| `input_bundle` | Milestone 5 (bundle that fits their crops) |
| `programme` | Your recommendation |

### 🧩 Milestone 6: Build the strategy table
**Difficulty:** ⭐⭐⭐  |  **Time:** ~30 min

Create a DataFrame `strategy` with **one row per farmer segment** and these columns:
`segment`, `share_%`, `key_constraint`, `product_focus`, `market_channel`, `input_bundle`, `programme`.

- `share_%` must come from `f_sizes` (numbers, rounded to 1 decimal).
- `product_focus` should name the **crop group that segment grows most** (use `exposure`): compute it, don't type it.
- All other text columns must be **non-empty** and **specific** (mention numbers or named products).

<details><summary>💡 Hint 1</summary>

`product_focus = exposure.idxmax(axis=1)` gives, for each segment, the crop group with the largest share.

</details>
<details><summary>💡 Hint 2</summary>

Constraints from profiles: subsistence → ~27% post-harvest loss, 14% co-op; horticulture → perishables, price volatility; commercial → storage & price risk; dairy/mixed → fodder & herd health.

</details>
<details><summary>💡 Hint 3</summary>

Input bundles from Milestone 5: maize planting (Hybrid Maize Seed+DAP+CAN) · crop protection (Pesticide+Sprayer+Gloves) · dairy (Dairy Meal+Mineral Lick+Acaricide) · potato (Potato Seed+DAP+Fungicide).

</details>

In [13]:
focus = exposure.idxmax(axis=1)
loss = loss_by_segment.round(0).astype(int)
text = {
    "Subsistence smallholders": ("~%d%% post-harvest loss, minimal fertiliser, few in co-ops" % loss["Subsistence smallholders"],
                                 "Sell via regional aggregation markets; join co-ops for bulk sales",
                                 "Starter pack: Hybrid Maize Seed + DAP (+ CAN top-dress)",
                                 "Hermetic storage bags + starter input credit + co-op recruitment"),
    "Intensive horticulture farmers": ("Perishable crops, volatile prices, depend on irrigation",
                                       "Urban wholesale hubs with cold storage (Nairobi, Mombasa, Nakuru hubs)",
                                       "Crop protection: Pesticide + Knapsack Sprayer + Gloves; Vegetable Seeds + Drip Kit",
                                       "Price-information SMS, cold-chain at hubs, irrigation maintenance"),
    "Commercial grain farmers": ("Large harvest peaks, storage and price risk",
                                 "Regional aggregation markets and warehouse receipts",
                                 "Maize planting: Hybrid Maize Seed + DAP + CAN (+ Herbicide)",
                                 "Warehouse receipt system, forward contracts, input bulk-buying"),
    "Dairy / mixed co-operative farmers": ("Fodder and herd-health productivity; already co-op organised",
                                           "Co-op cooling plants; regional aggregation markets",
                                           "Dairy: Dairy Meal + Mineral Lick + Acaricide",
                                           "Fodder and herd-health training delivered through the co-op"),
}
strategy = pd.DataFrame({"segment": f_sizes.index, "share_%": f_sizes.round(1).values})
strategy["key_constraint"] = [text[s][0] for s in strategy["segment"]]
strategy["product_focus"] = [focus[s] for s in strategy["segment"]]
strategy["market_channel"] = [text[s][1] for s in strategy["segment"]]
strategy["input_bundle"] = [text[s][2] for s in strategy["segment"]]
strategy["programme"] = [text[s][3] for s in strategy["segment"]]
pd.set_option("display.max_colwidth", 80); display(strategy)

,segment,share_%,key_constraint,product_focus,market_channel,input_bundle,programme
0,Subsistence smallholders,36.0,"~27% post-harvest loss, minimal fertiliser, few in co-ops",Storable staple grains,Sell via regional aggregation markets; join co-ops for bulk sales,Starter pack: Hybrid Maize Seed + DAP (+ CAN top-dress),Hermetic storage bags + starter input credit + co-op recruitment
1,Dairy / mixed co-operative farmers,28.6,Fodder and herd-health productivity; already co-op organised,Fodder (not traded),Co-op cooling plants; regional aggregation markets,Dairy: Dairy Meal + Mineral Lick + Acaricide,Fodder and herd-health training delivered through the co-op
2,Intensive horticulture farmers,19.4,"Perishable crops, volatile prices, depend on irrigation",Perishable vegetables & tropical fruit,"Urban wholesale hubs with cold storage (Nairobi, Mombasa, Nakuru hubs)",Crop protection: Pesticide + Knapsack Sprayer + Gloves; Vegetable Seeds + Dr...,"Price-information SMS, cold-chain at hubs, irrigation maintenance"
3,Commercial grain farmers,16.0,"Large harvest peaks, storage and price risk",Storable staple grains,Regional aggregation markets and warehouse receipts,Maize planting: Hybrid Maize Seed + DAP + CAN (+ Herbicide),"Warehouse receipt system, forward contracts, input bulk-buying"


In [14]:
# ✔ Self-check (run me after you've written your answer)
check('one row per segment', lambda: len(strategy) == f_k)
check('all required columns exist', lambda: {'segment', 'share_%', 'key_constraint', 'product_focus', 'market_channel', 'input_bundle', 'programme'} <= set(strategy.columns))
check('shares sum to about 100', lambda: abs(strategy['share_%'].sum() - 100) < 1)
check('no empty cells', lambda: (strategy.astype(str).apply(lambda col: col.str.len()) > 2).all().all())
check('product_focus is computed from the data (is a real crop group)', lambda: set(strategy['product_focus']) <= set(exposure.columns))

✅ one row per segment
✅ all required columns exist
✅ shares sum to about 100
✅ no empty cells
✅ product_focus is computed from the data (is a real crop group)


## Milestone 7 · The one-page brief ✍️

Dr. Kariuki gave you **one page**. That means ~400–500 words. Every sentence must earn its place.

### How to structure it (the "pyramid"):
1. **Answer first.** What should the Committee *do*? (1–2 sentences)
2. **Three findings**, each with a **number** and a **"so what?"**
3. **Where the money should go** (a table or 3–4 bullets)
4. **Why you can trust this** (1 short paragraph: methods in plain words, checks you did)
5. **Limits** and **next step**

### Style rules
- **No jargon**: not "DBSCAN" but "markets too isolated to be served by any hub".
- **Numbers with comparisons**: "27% vs 9–14%", not "high".
- **Include at least one result that surprised you**: and say it plainly.
- **One decision requested.**

### ✍️ The Agricultural Intelligence Brief (one page)
*(Example answer. Yours will differ in wording and possibly in numbers.)*

**To:** Dr. Wanjiku Kariuki  **From:** Lead Data Scientist  **Re:** Where next year's agricultural support budget should go

**Recommendation.** Fund **post-harvest storage and starter inputs for subsistence smallholders first**, then regional hubs in seven well-populated market regions, and run a *separate* low-cost mobile-services pilot for seven isolated markets. Do **not** fund blanket cold-chain as the main loss-reduction tool.

**Three findings**
1. **Four types of farmer, and one of them is losing the most.** Subsistence smallholders (36% of farmers) lose **~27% of their harvest after harvesting**, versus 9–14% for the other three types, and only 14% belong to a co-operative. *So what?* This is the largest group with the largest recoverable loss.
2. **Perishable crops are *not* where the losses are (a surprise).** Farmers growing tomatoes, kale and similar crops lose ~14% vs ~18% for everyone else. They are mostly irrigated, co-op-organised horticulture farmers close to markets. *So what?* A blanket cold-chain subsidy would miss the problem; target **by farmer type**, and put cold storage only at the horticulture-serving hubs.
3. **Seven market regions can each be served by one hub; seven markets cannot.** Markets cluster around Nairobi, Mombasa, Kisumu, Nakuru, Meru, Nyeri and Eldoret–Kitale, with average distances of ~14–48 km to a proposed hub. Seven markets (7%) average ~200 km from a city, have poor roads and **no cold storage**. *So what?* A fixed hub would serve almost nobody there: use mobile aggregation and price-information SMS.

**Where the money should go**

| Lever | Focus | Target group | Evidence |
|---|---|---|---|
| Farmer programmes | Hermetic storage + starter credit + co-op recruitment | Subsistence smallholders (36%) | 27% loss; 14% co-op membership |
| Market infrastructure | One hub in each of 7 regions; mobile services for 7 isolated markets | Regional hubs; remote markets | Density-based market clusters |
| Commodity policy | Cold-chain & price SMS for perishables; warehouse receipts for grains | Horticulture (19%); commercial grain (16%) | Perishable group volatility ≈ 0.4; grain shelf life ≈ 1 year |
| Input-supply | Three bundles: Hybrid Maize Seed + DAP + CAN; Dairy Meal + Mineral Lick + Acaricide; Pesticide + Sprayer + Gloves | Grain/subsistence; dairy co-ops; horticulture | Bundles co-purchased 3-9× more often than chance (lift 3-9) |

**Why you can trust it.** Farmer and commodity groups were found by two independent methods that agreed closely, and were stable across random samples. Market regions were found using real distances in kilometres. Input bundles come from 3,000 receipts and each co-occurs at least 3× more than chance.

**Limits and next step.** Data are a single registry snapshot (synthetic in this exercise); losses are self-reported. We link farmers to commodities by primary crop only. Next step: pilot the storage programme in two counties with a control group before national roll-out.

**Decision requested.** Approve the storage-and-starter-input pilot for subsistence smallholders and hub feasibility studies in the seven regions.

## 🎤 Optional: the 5-minute talk
Present your brief to classmates: **1 slide per finding**, one slide for the strategy table, one for limits. Rehearse the *surprise*. Peers should be able to answer: *"What is the decision, and what is the evidence?"*

## 🔍 Peer-review checklist
Swap notebooks with a classmate. Tick each box (and write one sentence of feedback for any ✗):

- [ ] Features chosen sensibly; revenue/ids/labels excluded; scaling and log-transforms explained
- [ ] *k* justified with more than one kind of evidence
- [ ] DBSCAN used radians and haversine correctly
- [ ] The **hypothesis was tested honestly**
- [ ] Association rules filtered by **lift**, not just confidence
- [ ] Strategy table is specific and traceable to evidence
- [ ] The brief fits on one page and leads with the answer
- [ ] A surprising finding is stated plainly
- [ ] Limitations are real, not boilerplate

## 🚀 Beyond the course
- **Re-run it yearly**; segments drift. How would you detect when your clusters go stale?
- **Add time**: seasonal price series per commodity would enable *time-series clustering*.
- **Go supervised**: use your segments as a target and build a classifier for new farmers (Phase 6!).
- **Try HDBSCAN, Gaussian mixtures, UMAP** and see what changes.

### 🎓 Congratulations!
You can now take an unlabelled table (or a pile of receipts), find the hidden structure, test whether it is real, name it, and persuade a decision-maker to act on it. That is **unsupervised learning in practice**.